# P1-27d · UEF breast masks: is the pectoral muscle inside? (CPU, ~3 min; masks only, no images)
Can run while P1-27c is downloading (separate session). Reads `P1_v2/e3_raw/train.zip` and `P1_v2/e3/vindr_meta/breast-level_annotations.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
import os, zipfile, numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
OUT = '/content/drive/MyDrive/P1_v2'; E3 = f'{OUT}/e3'; L = '/content/uef'
if not os.path.isdir(L): zipfile.ZipFile(f'{OUT}/e3_raw/train.zip').extractall(L)
MSK = [r for r, _, fs in os.walk(L) if os.path.basename(r).lower() == 'breast_masks'][0]
BL = pd.read_csv(f'{E3}/vindr_meta/breast-level_annotations.csv').drop_duplicates('image_id').set_index('image_id')
rows = []
for f in sorted(os.listdir(MSK)):
    iid = os.path.splitext(f)[0]
    if iid not in BL.index: continue
    m = cv2.imread(f'{MSK}/{f}', 0) > 127; H, W = m.shape
    left = m[:, :W // 2].sum() >= m[:, W // 2:].sum()          # chest wall on the side with more mask
    corner = m[: H // 8, : W // 6] if left else m[: H // 8, -W // 6:]
    rows.append(dict(image_id=iid, view=BL.loc[iid, 'view_position'], laterality=BL.loc[iid, 'laterality'], h=H, w=W,
                     mask_frac=float(m.mean()), corner_cover=float(corner.mean())))
T = pd.DataFrame(rows); T.to_csv(f'{E3}/uef_mask_check.csv', index=False)
print(T.groupby('view')[['mask_frac', 'corner_cover']].describe().round(3).to_string())
sel = pd.concat([T[T.view == 'MLO'].sample(24, random_state=0), T[T.view == 'CC'].sample(12, random_state=0)])
fig, ax = plt.subplots(4, 9, figsize=(18, 10))
for a, (_, r) in zip(ax.ravel(), sel.iterrows()):
    a.imshow(cv2.imread(f'{MSK}/{r.image_id}.jpg', 0), cmap='gray'); a.set_title(f'{r.view} {r.laterality} {r.corner_cover:.2f}', fontsize=8); a.axis('off')
plt.tight_layout(); os.makedirs(f'{E3}/sheets', exist_ok=True); plt.savefig(f'{E3}/sheets/uef_masks_only.png', dpi=60); print('DONE')